# SolarVoix — Traitement incrémental automatique

**Notebook Fabric :** `NB_Traitement_Incremental`

**Lakehouse requis :** attachez `LH_SolarVoix` comme Lakehouse par défaut.

Le paramètre `call_file_path` accepte un chemin explicite pour les tests ou `AUTO` pour le déclenchement OneLake.

> **Modèle autonome :** le notebook charge `sentiment_tfidf.joblib` depuis OneLake. S'il n'existe pas encore, il l'entraîne automatiquement à partir des tables Bronze. Aucun modèle Hugging Face ou PyTorch n'est téléchargé.


## Cellule 0 — Installation des dépendances


In [ ]:
# === CELLULE 0 : Installation des dépendances ===

import subprocess
import sys

packages = ["scikit-learn", "joblib"]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", *packages],
    check=True,
)

print("✅ Packages installés avec succès")

# Le redémarrage rend les packages nouvellement installés disponibles.
# Aucun autre code ne doit être placé après cette instruction dans la cellule.
import notebookutils
notebookutils.session.restartPython()


## Cellule 1 — Paramètre d'entrée


In [ ]:
# === CELLULE 1 : Paramètre d'entrée ===
#
# IMPORTANT : cette cellule doit rester marquée comme "Parameters".
# Fabric insère la valeur transmise par le Pipeline juste après
# l'exécution de cette cellule.

call_file_path = "AUTO"


## Cellule 1 bis — Résolution du fichier à traiter


In [ ]:
# === CELLULE 1 BIS : Résolution du fichier à traiter ===
#
# Fabric a déjà injecté ici la valeur transmise par le Pipeline.
# Un chemin explicite est conservé ; AUTO recherche un appel absent de Silver.

from notebookutils import mssparkutils

if call_file_path == "AUTO":
    try:
        fichiers_dossier = mssparkutils.fs.ls("Files/audio_calls/")
        ids_dossier = {
            fichier.name.removesuffix(".txt")
            for fichier in fichiers_dossier
            if fichier.name.endswith(".txt")
        }

        df_silver = spark.table("silver_appels_enrichis")
        ids_silver = {
            ligne.call_id
            for ligne in df_silver.select("call_id").collect()
        }
        nouveaux_ids = ids_dossier - ids_silver

        print(f"📊 Fichiers présents dans audio_calls : {len(ids_dossier)}")
        print(f"📊 Appels déjà traités dans Silver    : {len(ids_silver)}")
        print(f"📊 Nouveaux appels à traiter          : {len(nouveaux_ids)}")

        if not nouveaux_ids:
            raise RuntimeError(
                "Aucun nouveau fichier n'est disponible dans Files/audio_calls/."
            )

        call_id_temp = sorted(nouveaux_ids)[0]
        call_file_path = f"Files/audio_calls/{call_id_temp}.txt"
        print(
            "🚀 Nouveau fichier détecté automatiquement "
            f"→ {call_file_path}"
        )

    except Exception as e:
        raise RuntimeError(
            f"Échec de la détection automatique : {e}"
        ) from e

else:
    print(
        "🧪 Chemin explicite reçu du Pipeline : "
        f"{call_file_path}"
    )

print(f"Paramètre final utilisé : call_file_path = {call_file_path}")


## Cellule 2 — Extraction du call_id


In [ ]:
# === CELLULE 2 : Extraction du call_id ===
#
# OBJECTIF : Dériver le call_id depuis le chemin de fichier reçu en paramètre.
#
# Exemple : "Files/audio_calls/CALL_0121.txt"
#           → call_id = "CALL_0121"
#
# On utilise os.path.basename pour extraire le nom du fichier,
# puis str.replace pour supprimer l'extension.
# Robuste même si le chemin contient des sous-dossiers supplémentaires.

import os
import re

nom_fichier = os.path.basename(call_file_path)   # "CALL_0121.txt"
call_id = nom_fichier.replace(".txt", "")         # "CALL_0121"

# Validation : le call_id doit correspondre au pattern CALL_XXXX
if not re.match(r"^CALL_\d{4}$", call_id):
    raise ValueError(f"Format call_id invalide : '{call_id}'. Attendu: CALL_XXXX")

print(f"call_id extrait : {call_id}")


## Cellule 3 — Lecture et enrichissement


In [ ]:
# === CELLULE 3 : Traitement d'un seul appel (Version corrigée) ===

from pathlib import Path
import joblib
import pyspark.sql.functions as F

# ====================== VARIABLES PAR DÉFAUT ======================
# Sécurité : valeurs par défaut si la cellule est exécutée isolément
try:
    call_file_path
except NameError:
    call_file_path = "Files/audio_calls/CALL_0001.txt"

try:
    call_id
except NameError:
    # On extrait le nom du fichier comme call_id (ex: CALL_0001)
    call_id = call_file_path.split("/")[-1].replace(".txt", "")

# ====================== LECTURE DU FICHIER ======================
try:
    # mssparkutils.fs.head retourne déjà un str (UTF-8)
    transcription_text = mssparkutils.fs.head(call_file_path, 10000).strip()

    if not transcription_text:
        raise ValueError("Le fichier est vide ou ne contient que des espaces.")

    print(f"Fichier lu avec succès : {len(transcription_text)} caractères")
    print(f"Call ID : {call_id} | Chemin : {call_file_path}")

except Exception as e:
    raise FileNotFoundError(f"Impossible de lire le fichier {call_file_path} : {e}") from e

# ====================== CHARGEMENT / CRÉATION DU MODÈLE ======================
MODEL_PATH = "/lakehouse/default/Files/models/sentiment_tfidf.joblib"

if not Path(MODEL_PATH).exists():
    print("Modèle absent : entraînement local automatique depuis les tables Bronze...")
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import Pipeline

    tables_requises = {"bronze_transcriptions", "bronze_calls_metadata"}
    tables_absentes = [t for t in tables_requises if not spark.catalog.tableExists(t)]
    if tables_absentes:
        raise RuntimeError(
            "Impossible de créer le modèle : tables Bronze absentes : "
            + ", ".join(sorted(tables_absentes))
        )

    df_entrainement = (
        spark.table("bronze_transcriptions").alias("t")
        .join(spark.table("bronze_calls_metadata").alias("m"), "call_id")
        .select("transcription_text", "sentiment_reel")
        .dropna()
    )
    lignes_entrainement = df_entrainement.collect()
    if not lignes_entrainement:
        raise RuntimeError("Impossible de créer le modèle : aucune donnée Bronze étiquetée.")

    def normaliser_label(label: str) -> str:
        if label in {"tres_negatif", "negatif"}:
            return "negatif"
        if label == "positif":
            return "positif"
        return "neutre"

    textes_entrainement = [r.transcription_text for r in lignes_entrainement]
    labels_entrainement = [normaliser_label(r.sentiment_reel) for r in lignes_entrainement]
    if len(set(labels_entrainement)) < 2:
        raise RuntimeError("Impossible de créer le modèle : au moins deux classes sont nécessaires.")

    modele_sentiment = Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True, strip_accents="unicode", ngram_range=(1, 2), min_df=1
        )),
        ("classifier", LogisticRegression(
            max_iter=1000, class_weight="balanced", random_state=42
        )),
    ])
    modele_sentiment.fit(textes_entrainement, labels_entrainement)
    Path(MODEL_PATH).parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(modele_sentiment, MODEL_PATH)
    print(f"Modèle local entraîné et sauvegardé : {MODEL_PATH}")
else:
    print(f"Chargement du modèle local depuis OneLake : {MODEL_PATH}")
    modele_sentiment = joblib.load(MODEL_PATH)

# ====================== FONCTION PRINCIPALE ======================
def enrichir_appel(call_id: str, texte: str, duree_sec: int = 180) -> dict:
    """
    Applique l'analyse IA et calcule le score churn pour un appel unique.
    Retourne un dictionnaire prêt pour la table silver_appels_enrichis.
    """
    # Sentiment IA
    # Conversion explicite de numpy.str_ vers str Python pour Spark.
    sentiment_ia = str(modele_sentiment.predict([texte])[0])
    probabilites = modele_sentiment.predict_proba([texte])[0]
    score_sentiment = round(float(max(probabilites)), 4)

    # Détection d'intention (mots-clés)
    KEYWORDS_INTENT = {
        "resiliation":         ["résilie", "résiliation", "annuler", "annule", "partir", "quitter"],
        "menace_juridique":    ["avocat", "plainte", "dgccrf", "médiateur", "tribunal", "juridique"],
        "churn_imminent":      ["jamais", "arnaque", "honte", "incompétent", "remboursement total"],
        "plainte_technique":   ["panne", "ne fonctionne pas", "cassé", "défaillant", "réparer"],
        "plainte_sav":         ["rappelle pas", "personne ne", "inacceptable", "scandaleux"],
        "question_facturation":["facture", "tva", "remboursement", "devis", "tarif", "prix"],
        "suivi_commande":      ["livraison", "délai", "date", "quand", "attends"],
        "satisfaction":        ["merci", "excellent", "parfait", "satisfait", "recommande"],
        "information":         ["information", "renseignement", "savoir", "comprendre"],
    }

    texte_lower = texte.lower()
    intent_detecte = "autre"
    priorite = ["resiliation", "menace_juridique", "churn_imminent",
                "plainte_technique", "plainte_sav", "question_facturation",
                "suivi_commande", "satisfaction", "information"]
    for intent in priorite:
        if any(kw in texte_lower for kw in KEYWORDS_INTENT[intent]):
            intent_detecte = intent
            break

    # Même formule que dans la Partie 1 afin qu'un même appel conserve le même score.
    score = 0
    if sentiment_ia == "negatif":
        score += 40

    mots_resil = ["résilie", "résiliation", "annuler", "quitter", "partir", "dernier"]
    if any(m in texte_lower for m in mots_resil):
        score += 30

    mots_menace = ["avocat", "plainte", "dgccrf", "médiateur", "tribunal", "assurance"]
    if any(m in texte_lower for m in mots_menace):
        score += 20

    mots_plainte = ["panne", "ne fonctionne pas", "inacceptable", "scandaleux",
                    "inadmissible", "trois fois", "cinq fois", "rappelle pas"]
    if any(m in texte_lower for m in mots_plainte):
        score += 15

    if duree_sec > 300:
        score += 10

    score_risque_churn = min(score, 100)

    return {
        "call_id":            call_id,
        "sentiment_ia":       sentiment_ia,
        "score_sentiment":    score_sentiment,
        "intent_detecte":     intent_detecte,
        "score_risque_churn": score_risque_churn,
        "alerte_critique":    score_risque_churn >= 50,
        "duree_sec":          duree_sec,           # utile de le garder
    }


# ====================== EXÉCUTION ======================
resultat = enrichir_appel(call_id, transcription_text, duree_sec=180)

print(f"\n✅ Résultat de l'enrichissement de l'appel :")
for k, v in resultat.items():
    print(f"   {k:20} : {v}")


## Cellule 4 — UPSERT Silver et alerte idempotente


In [ ]:
# === CELLULE 4 : UPSERT dans la table Silver (Delta MERGE) ===
#
# OBJECTIF : Mettre à jour la table silver_appels_enrichis pour le call_id traité.
#            Si le call_id existe déjà → UPDATE (ré-analyse).
#            Si le call_id est nouveau → INSERT.
#
# MERGE INTO est la commande Delta pour les UPSERTS.
# Plus efficace que DELETE + INSERT car Delta ne réécrit que les fichiers concernés.
#
# Note : mssparkutils.notebook.exit() retourne un JSON au Pipeline Data Factory.
# Ce JSON contient le score churn ET le chemin du fichier réellement traité.
# Le Pipeline utilise json(...).score pour la condition d'alerte
# et json(...).fichier pour identifier le fichier dans les logs Monitor.

from datetime import datetime
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType,
    IntegerType, BooleanType, TimestampType,
)

# Schéma explicite : évite l'inférence de types lors d'un appel Pipeline.
schema_nouveau = StructType([
    StructField("call_id", StringType(), False),
    StructField("sentiment_ia", StringType(), False),
    StructField("score_sentiment", DoubleType(), False),
    StructField("intent_detecte", StringType(), False),
    StructField("score_risque_churn", IntegerType(), False),
    StructField("alerte_critique", BooleanType(), False),
    StructField("date_mise_a_jour", TimestampType(), False),
])

ligne_nouvelle = (
    str(resultat["call_id"]),
    str(resultat["sentiment_ia"]),
    float(resultat["score_sentiment"]),
    str(resultat["intent_detecte"]),
    int(resultat["score_risque_churn"]),
    bool(resultat["alerte_critique"]),
    datetime.now(),
)
df_nouveau = spark.createDataFrame([ligne_nouvelle], schema=schema_nouveau)
df_nouveau.createOrReplaceTempView("staging_appel")

# MERGE INTO : UPSERT sur call_id
spark.sql("""
    MERGE INTO silver_appels_enrichis AS target
    USING staging_appel AS source
    ON target.call_id = source.call_id
    WHEN MATCHED THEN
        UPDATE SET
            target.sentiment_ia       = source.sentiment_ia,
            target.score_sentiment    = source.score_sentiment,
            target.intent_detecte     = source.intent_detecte,
            target.score_risque_churn = source.score_risque_churn,
            target.alerte_critique    = source.alerte_critique
    WHEN NOT MATCHED THEN
        INSERT (call_id, sentiment_ia, score_sentiment,
                intent_detecte, score_risque_churn, alerte_critique)
        VALUES (source.call_id, source.sentiment_ia, source.score_sentiment,
                source.intent_detecte, source.score_risque_churn, source.alerte_critique)
""")

print(f"MERGE terminé pour {call_id}")
print(f"Score churn : {resultat['score_risque_churn']} — Alerte : {resultat['alerte_critique']}")

# -------------------------------------------------------
# ÉCRITURE DANS LA TABLE alertes_critiques (si score ≥ 50)
# -------------------------------------------------------
# Cette table est la seule "notification" nécessaire : elle est
# requêtable en SQL, visible dans Power BI, et ne dépend d'aucun
# service externe (pas de Teams, pas d'email, pas de webhook).
#
# Le Pipeline lit le score via mssparkutils.notebook.exit()
# pour afficher le statut dans le panneau Monitor de Fabric.

if resultat["alerte_critique"]:
    df_alerte = spark.createDataFrame([{
        "call_id":            resultat["call_id"],
        "score_risque_churn": int(resultat["score_risque_churn"]),
        "sentiment_ia":       resultat["sentiment_ia"],
        "intent_detecte":     resultat["intent_detecte"],
        "statut":             "NON_TRAITEE",
        "date_alerte":        datetime.now(),
    }])
    df_alerte.createOrReplaceTempView("staging_alerte")

    if not spark.catalog.tableExists("alertes_critiques"):
        df_alerte.write.mode("overwrite").format("delta").saveAsTable("alertes_critiques")
        print(f"🚨 Table alertes_critiques créée pour {call_id}")
    else:
        spark.sql("""
            MERGE INTO alertes_critiques AS target
            USING staging_alerte AS source
            ON target.call_id = source.call_id
            WHEN MATCHED THEN
                UPDATE SET
                    target.score_risque_churn = source.score_risque_churn,
                    target.sentiment_ia       = source.sentiment_ia,
                    target.intent_detecte     = source.intent_detecte,
                    target.statut             = source.statut,
                    target.date_alerte        = source.date_alerte
            WHEN NOT MATCHED THEN
                INSERT (call_id, score_risque_churn, sentiment_ia,
                        intent_detecte, statut, date_alerte)
                VALUES (source.call_id, source.score_risque_churn, source.sentiment_ia,
                        source.intent_detecte, source.statut, source.date_alerte)
        """)
        print(f"🚨 Alerte mise à jour dans alertes_critiques pour {call_id}")
else:
    print(f"✅ Appel {call_id} traité sans alerte (score = {resultat['score_risque_churn']})")

# Retourne un JSON au Pipeline → visible dans Monitor → panneau Output
# Contient le score ET le fichier réellement traité (détecté automatiquement)
# Le Pipeline lit json(...).score pour la condition et json(...).fichier pour les logs
import json
mssparkutils.notebook.exit(json.dumps({
    "score": resultat["score_risque_churn"],
    "fichier": call_file_path
}))
